# Arena 3D Reconstruction with Gaussian Splatting

Reconstruct a 10–15m arena from 91 photos using COLMAP + 3D Gaussian Splatting,
then compress the model for lightweight local viewing.

## Pipeline Overview (run cells in order)

| Step | Cell | What it does | Time | GPU? |
|------|------|-------------|------|------|
| 1 | **Cell 1** | Mount Google Drive + choose session | ~30s | No |
| 2 | **Cell 2** | Install COLMAP, PyTorch, CUDA extensions | ~3 min | No* |
| 3 | **Cell 3** | Download 91 arena photos from GitHub | ~2 min | No |
| 4 | **Cell 4A–4D** | COLMAP SfM: features → matching → reconstruction → merge | ~25 min | No |
| 5 | **Cell 5** | *OR* download pre-computed COLMAP data (skip step 4) | ~1 min | No |
| 6 | **Cell 6** | Convert COLMAP to 3DGS format (SIMPLE_RADIAL → PINHOLE) | ~1 min | No |
| 7 | **Cell 7A–7B** | Train 3D Gaussian Splatting (enhanced, gsplat-based) | 7–30 min | **Yes (T4+)** |
| 8 | **Cell 8A** | Export final point cloud PLY | ~1 min | No |
| 9 | **Cell 8B** | Validate PLY for Unity + viewing options | ~1 min | No |
| 10 | **Cell 8C** | Compress model for local decompression viewer | ~1 min | No |

\* PyTorch installs but CUDA extensions only build with a GPU runtime.

## Requirements

- **Runtime:** Go to Runtime → Change runtime type → **T4 GPU** (or any GPU)
- **Google Drive:** ~2 GB free for checkpoints
- **Internet:** Stable connection (~500 MB downloads total)

## Session Management

This notebook saves its progress to Google Drive. If your session disconnects
(reconnect, timeout, etc.), re-open the notebook and **run Cell 1** — it will
ask if you want to continue from where you left off.

## Outputs

| File | Location | Size |
|------|----------|------|
| Final model (PLY) | `arena_3dgs_pointcloud.ply` (also on Drive) | ~100–500 MB |
| Compressed model (.splat) | `arena_3dgs_compressed.splat` | ~10–50 MB |
| Training output (PLY) | `output/arena_3dgs/arena_3dgs.ply` | ~100–500 MB |
| COLMAP database | `MyDrive/arena_3dgs/database.db` | ~100 MB |
| Sparse model | `MyDrive/arena_3dgs/sparse_model` | ~10 MB |


In [ ]:
#@title === 1. Mount Drive + Session Management ===
import os, json, datetime

DRIVE_PATH = "/content/drive/MyDrive/arena_3dgs"
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f"Checkpoints will be saved to: {DRIVE_PATH}")

# ═════════════════════════════════════════════════════════════════
#  Session state module
# ═════════════════════════════════════════════════════════════════
SESSION_PATH = os.path.join(DRIVE_PATH, "session_state.json")

def load_session():
    if os.path.exists(SESSION_PATH):
        with open(SESSION_PATH) as f:
            return json.load(f)
    return {"steps": {}, "params": {}, "created_at": None, "updated_at": None}

def save_session(session):
    session["updated_at"] = datetime.datetime.now().isoformat()
    with open(SESSION_PATH, "w") as f:
        json.dump(session, f, indent=2)
    return session

def mark_step(step_name):
    session = load_session()
    session["steps"][step_name] = True
    save_session(session)
    print(f"  [session] Step '{step_name}' completed")

def is_step_done(step_name):
    session = load_session()
    return session["steps"].get(step_name, False)

def set_param(key, value):
    session = load_session()
    session["params"][key] = value
    save_session(session)

def get_param(key, default=None):
    session = load_session()
    return session["params"].get(key, default)

def reset_session():
    if os.path.exists(SESSION_PATH):
        os.remove(SESSION_PATH)
    session = {"steps": {}, "params": {}, "created_at": datetime.datetime.now().isoformat()}
    save_session(session)
    print("  [session] Fresh session started — all steps will run from scratch.")

def checkpoint_path(name):
    return os.path.join(DRIVE_PATH, name)

def save_to_drive(src, name):
    dst = checkpoint_path(name)
    if os.path.exists(src):
        !cp -r "{src}" "{dst}"
        print(f"  Saved checkpoint: {name}")

def restore_from_drive(name, dst):
    src = checkpoint_path(name)
    if os.path.exists(src):
        !cp -r "{src}" "{dst}"
        print(f"  Restored checkpoint: {name} -> {dst}")
        return True
    return False

def checkpoint_exists(name):
    return os.path.exists(checkpoint_path(name))

# ═════════════════════════════════════════════════════════════════
#  Session choice
# ═════════════════════════════════════════════════════════════════
existing_session = load_session()
has_previous = existing_session["created_at"] is not None

print()
print("=" * 60)
if has_previous:
    done_steps = [k for k, v in existing_session["steps"].items() if v]
    total_params = existing_session.get("params", {})
    print(f"  Previous session found (created: {existing_session['created_at'][:19]})")
    print(f"  Completed steps: {len(done_steps)}")
    if done_steps:
        print(f"    {', '.join(done_steps)}")
    if total_params:
        print(f"  Parameters: {json.dumps(total_params, default=str)}")
    print("=" * 60)
    choice = input("\nContinue from previous session? [Y/n]: ").strip().lower() or "y"
    if choice == "y":
        print("\n  [session] Continuing previous session. Completed steps will be skipped.")
    else:
        print("\n  [session] Starting fresh. Previous checkpoints on Drive remain untouched.")
        reset_session()
        existing_session = load_session()
else:
    print("  No previous session found. Starting fresh.")
    print("=" * 60)
    reset_session()
    existing_session = load_session()

print(f"\nSession ID: {existing_session['created_at'][:19]}")
mark_step("drive_mounted")


In [ ]:
#@title === 2. Install Dependencies (~3 min, idempotent) ===
import os, subprocess, atexit, sys

colmap_available = bool(os.popen("which colmap 2>/dev/null").read().strip())
skip_deps = is_step_done("deps_installed") and colmap_available

if skip_deps:
    print("Dependencies already installed (session state). Skipping.")
else:
    if is_step_done("deps_installed"):
        print("Session says deps installed but colmap not found. Re-installing.")

    # ── Debian packages + pip installs (gated by Drive checkpoint) ──
    DRV = checkpoint_path("deps_installed")
    colmap_missing = not colmap_available
    if not os.path.exists(DRV) or colmap_missing:
        print("[1/4] Installing COLMAP + display deps...")
        !apt-get update -qq && apt-get install -y -qq colmap xvfb libgl1-mesa-glx libglib2.0-0
        !colmap version 2>&1 | head -1

        print("[2/4] Installing PyTorch...")
        !pip install torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118 -q
        import torch
        cuda_info = f"CUDA: {torch.cuda.is_available()}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB" if torch.cuda.is_available() else "CUDA: False"
        print(f"  PyTorch {torch.__version__}, {cuda_info}")

        print("[3/4] Installing Python packages...")
        !pip install plyfile numpy pillow opencv-python-headless tqdm gsplat scipy -q

        print("[4/4] Verifying GPU...")
        !touch "{DRV}"
        print("\nSystem deps installed!")
    else:
        print("System deps already installed (Drive marker found).")

    # ── Download enhanced training script from GitHub ──
    import urllib.request
    SCRIPTS_DIR = "/content/scripts"
    os.makedirs(SCRIPTS_DIR, exist_ok=True)
    ENHANCED_PY = os.path.join(SCRIPTS_DIR, "train_3dgs_enhanced.py")
    if not os.path.exists(ENHANCED_PY):
        print("Downloading enhanced training script from GitHub...")
        url = ("https://raw.githubusercontent.com/"
               "kaarthik-balakrishnan/arena-3dgs/main/scripts/train_3dgs_enhanced.py")
        urllib.request.urlretrieve(url, ENHANCED_PY)
        print("  Done.")
    else:
        print("Enhanced training script already downloaded.")
    sys.path.insert(0, SCRIPTS_DIR)

    mark_step("deps_installed")

# ── Virtual display for COLMAP (always runs after deps) ──
DISPLAY_NUM = 99
os.environ["DISPLAY"] = f":{DISPLAY_NUM}"
os.environ["QT_QPA_PLATFORM"] = "offscreen"
LOCK_FILE = f"/tmp/.X{DISPLAY_NUM}-lock"
if not os.path.exists(LOCK_FILE):
    xvfb_proc = subprocess.Popen(
        ["Xvfb", f":{DISPLAY_NUM}", "-screen", "0", "1024x768x24"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    atexit.register(lambda: xvfb_proc.terminate())
    print(f"Virtual display :{DISPLAY_NUM} started (PID {xvfb_proc.pid})")
else:
    print(f"Virtual display :{DISPLAY_NUM} already running.")

print("\nAll dependencies ready!")

---
## Step 3: Images (2 min, skips if already downloaded)
---


In [ ]:
#@title === 3: Download Images from GitHub (~2 min, idempotent) ===
import os, requests

if is_step_done("images_downloaded"):
    print("Images already downloaded (session state). Skipping.")
else:
    INPUT_DIR = "/content/gaussian-splatting/input"
    os.makedirs(INPUT_DIR, exist_ok=True)

    existing = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if len(existing) >= 91:
        print(f"{len(existing)} images already present. Skipping download.")
    else:
        GITHUB_REPO = "kaarthik-balakrishnan/arena-3dgs"
        api_url = f"https://api.github.com/repos/{GITHUB_REPO}/contents/splat-files-processed"
        resp = requests.get(api_url)
        if resp.status_code == 200:
            files_list = resp.json()
            for item in files_list:
                if item['name'].lower().endswith(('.jpg', '.jpeg', '.png')):
                    img_resp = requests.get(item['download_url'])
                    with open(os.path.join(INPUT_DIR, item['name']), 'wb') as f:
                        f.write(img_resp.content)
            imgs = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            print(f"Downloaded {len(imgs)} images from GitHub")
        else:
            print(f"GitHub API error ({resp.status_code}). Upload images manually to {INPUT_DIR}")

    imgs = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Total: {len(imgs)} images")
    set_param("num_images", len(imgs))
    mark_step("images_downloaded")


---
## COLMAP Step 4: Structure from Motion

Three modular cells: (A) features → (B) matching → (C) reconstruction.
Each saves checkpoints to Drive. If a cell crashes, re-run it.

**Alternatively**, skip to Step 5 and download pre-computed camera poses.
---


In [ ]:
#@title === 4A: Feature Extraction (~5 min, idempotent) ===
import os

if is_step_done("colmap_features"):
    print("Feature extraction already done (session state). Restoring from Drive and skipping.")
    DB_PATH = "/content/gaussian-splatting/sparse/database.db"
    restore_from_drive("database.db", DB_PATH)
else:
    # ── Check colmap is installed ──
    if not os.popen("which colmap 2>/dev/null").read().strip():
        raise RuntimeError(
            "colmap not found. Run Cell 2 first to install dependencies."
        )

    INPUT_DIR = "/content/gaussian-splatting/input"
    COLMAP_DIR = "/content/gaussian-splatting/sparse"
    os.makedirs(COLMAP_DIR, exist_ok=True)
    os.environ["QT_QPA_PLATFORM"] = "offscreen"
    DB_PATH = os.path.join(COLMAP_DIR, "database.db")

    # Restore from Drive if available
    if checkpoint_exists("database.db"):
        print("Database checkpoint found on Drive. Restoring...")
        restore_from_drive("database.db", DB_PATH)

    # Skip if features already extracted
    if os.path.exists(DB_PATH) and os.path.getsize(DB_PATH) > 1000000:
        print("Features already extracted. Skipping.")
    else:
        !colmap feature_extractor \
            --database_path {DB_PATH} \
            --image_path {INPUT_DIR} \
            --ImageReader.camera_model SIMPLE_RADIAL \
            --ImageReader.single_camera 1 \
            --SiftExtraction.use_gpu 0 \
            --SiftExtraction.max_num_features 8192 \
            --SiftExtraction.first_octave -1 \
            --SiftExtraction.peak_threshold 0.01
        print("\nSaving checkpoint to Drive...")
        save_to_drive(DB_PATH, "database.db")

    # Verify
    import sqlite3
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute("SELECT COUNT(*) FROM keypoints").fetchone()[0]
    print(f"  Keypoints tables: {rows}")
    conn.close()
    mark_step("colmap_features")


In [ ]:
#@title === 4B: Feature Matching (~5 min, idempotent) ===
import os, sqlite3

if is_step_done("colmap_matching"):
    print("Feature matching already done (session state). Skipping.")
else:
    # ── Check colmap is installed ──
    if not os.popen("which colmap 2>/dev/null").read().strip():
        raise RuntimeError(
            "colmap not found. Run Cell 2 first to install dependencies."
        )

    DB_PATH = "/content/gaussian-splatting/sparse/database.db"
    os.environ["QT_QPA_PLATFORM"] = "offscreen"

    # Check if matches already exist
    conn = sqlite3.connect(DB_PATH)
    match_count = conn.execute("SELECT COUNT(*) FROM matches").fetchone()[0]
    conn.close()

    if match_count > 1000:
        print(f"{match_count} match pairs already exist. Skipping.")
    else:
        print("\n=== Sequential Matching ===")
        !colmap sequential_matcher \
            --database_path {DB_PATH} \
            --SiftMatching.use_gpu 0 \
            --SequentialMatching.overlap 20
        save_to_drive(DB_PATH, "database.db")

        print("\n=== Exhaustive Matching ===")
        !colmap exhaustive_matcher \
            --database_path {DB_PATH} \
            --SiftMatching.use_gpu 0
        save_to_drive(DB_PATH, "database.db")

    # Verify
    conn = sqlite3.connect(DB_PATH)
    verified = conn.execute("SELECT COUNT(*) FROM two_view_geometries").fetchone()[0]
    print(f"  Verified: {verified} image pairs")
    conn.close()
    mark_step("colmap_matching")


In [ ]:
#@title === 4C: COLMAP Reconstruction (~10 min, idempotent) ===
import os, struct

if is_step_done("colmap_reconstruction"):
    print("COLMAP reconstruction already done (session state). Skipping.")
else:
    # ── Check colmap is installed ──
    if not os.popen("which colmap 2>/dev/null").read().strip():
        raise RuntimeError(
            "colmap not found. Run Cell 2 first to install dependencies."
        )

    INPUT_DIR = "/content/gaussian-splatting/input"
    DB_PATH = "/content/gaussian-splatting/sparse/database.db"
    COLMAP_DIR = "/content/gaussian-splatting/sparse"
    os.environ["QT_QPA_PLATFORM"] = "offscreen"
    MODEL_PATH = os.path.join(COLMAP_DIR, "0")

    # Skip if model already exists
    if os.path.exists(os.path.join(MODEL_PATH, "images.bin")):
        with open(os.path.join(MODEL_PATH, "images.bin"), "rb") as f:
            n = struct.unpack("Q", f.read(8))[0]
        print(f"Model already exists ({n} images). Skipping.")
    elif checkpoint_exists("sparse_model"):
        print("Restoring sparse model from Drive...")
        import shutil
        if os.path.exists(MODEL_PATH):
            shutil.rmtree(MODEL_PATH)
        restore_from_drive("sparse_model", MODEL_PATH)
    else:
        !colmap mapper \
            --database_path {DB_PATH} \
            --image_path {INPUT_DIR} \
            --output_path {COLMAP_DIR} \
            --Mapper.multiple_models 1 \
            --Mapper.max_num_models 50 \
            --Mapper.init_min_tri_angle 4 \
            --Mapper.init_min_num_inliers 15 \
            --Mapper.abs_pose_min_num_inliers 8 \
            --Mapper.ba_local_max_num_iterations 25 \
            --Mapper.ba_global_max_num_iterations 50

    # Find best model (most registered images)
    best_model = None
    best_n = 0
    for sub in sorted(os.listdir(COLMAP_DIR)):
        img_path = os.path.join(COLMAP_DIR, sub, "images.bin")
        if os.path.exists(img_path):
            with open(img_path, "rb") as f:
                n = struct.unpack("Q", f.read(8))[0]
            if n > best_n:
                best_n = n
                best_model = sub

    if best_model:
        print(f"\nBest model: sub={best_model}, images={best_n}")
        src_m = os.path.join(COLMAP_DIR, best_model)
        save_to_drive(src_m, "sparse_model")
        set_param("colmap_registered_images", best_n)
    else:
        print("No reconstruction produced. Use Step 5 for pre-computed data.")

    
    mark_step("colmap_reconstruction")

# ── Copy COLMAP data to Step 6's expected location ──
INPUT_SPARSE = "/content/gaussian-splatting/input/sparse/0"
os.makedirs(INPUT_SPARSE, exist_ok=True)
if best_model:
    model_dir = os.path.join(COLMAP_DIR, best_model)
    for fname in ["cameras.txt", "images.txt", "points3D.txt"]:
        txt_src = os.path.join(os.path.join(model_dir, "txt"), fname)
        if os.path.exists(txt_src):
            !cp "{txt_src}" "{INPUT_SPARSE}/{fname}"
        elif os.path.exists(os.path.join(model_dir, fname)):
            !cp "{os.path.join(model_dir, fname)}" "{INPUT_SPARSE}/{fname}"
        else:
            # Try converting to txt first if not already done
            !colmap model_converter --input_path "{model_dir}" --output_path "{os.path.join(model_dir, 'txt')}" --output_type TXT 2>/dev/null
            txt_src = os.path.join(os.path.join(model_dir, "txt"), fname)
            if os.path.exists(txt_src):
                !cp "{txt_src}" "{INPUT_SPARSE}/{fname}"

In [ ]:
#@title === 4D: Model Merging (~2 min, idempotent) ===
import os, subprocess, struct

if is_step_done("colmap_merged"):
    print("Model merging already done (session state). Skipping.")
else:
    # ── Check colmap is installed ──
    if not os.popen("which colmap 2>/dev/null").read().strip():
        raise RuntimeError(
            "colmap not found. Run Cell 2 first to install dependencies."
        )

    COLMAP_DIR = "/content/gaussian-splatting/sparse"
    MERGED_DIR = "/content/gaussian-splatting/sparse_merged"

    # Skip if already merged
    if os.path.exists(os.path.join(MERGED_DIR, "images.bin")):
        with open(os.path.join(MERGED_DIR, "images.bin"), "rb") as f:
            n = struct.unpack("Q", f.read(8))[0]
        print(f"Merged model already exists ({n} images). Skipping.")
    else:
        # Find all reconstructions with at least 5 images
        models = []
        for sub in sorted(os.listdir(COLMAP_DIR)):
            img_path = os.path.join(COLMAP_DIR, sub, "images.bin")
            if os.path.exists(img_path):
                with open(img_path, "rb") as f:
                    n = struct.unpack("Q", f.read(8))[0]
                if n >= 5:
                    models.append((sub, n))
                    print(f"  Found model {sub}: {n} images")

        if len(models) >= 2:
            print("\nAttempting to merge models...")
            models.sort(key=lambda x: -x[1])
            current = os.path.join(COLMAP_DIR, models[0][0])
            for i in range(1, min(3, len(models))):
                other = os.path.join(COLMAP_DIR, models[i][0])
                merge_out = f"/content/gaussian-splatting/merge_{i}"
                !mkdir -p "{merge_out}"
                result = subprocess.run(
                    ["colmap", "model_merger",
                     "--input_path1", current,
                     "--input_path2", other,
                     "--output_path", merge_out],
                    capture_output=True, text=True
                )
                if os.path.exists(os.path.join(merge_out, "images.bin")):
                    with open(os.path.join(merge_out, "images.bin"), "rb") as f:
                        n = struct.unpack("Q", f.read(8))[0]
                    print(f"  Merge with {models[i][0]} successful: {n} images")
                    current = merge_out
                else:
                    print(f"  Merge with {models[i][0]} failed (different coordinate systems)")
                    !rm -rf "{merge_out}"
            !cp -r "{current}" "{MERGED_DIR}"

        # Convert best to txt
        best_src = MERGED_DIR if os.path.exists(os.path.join(MERGED_DIR, "images.bin")) else                    os.path.join(COLMAP_DIR, models[0][0] if models else "0")
        txt_dir = os.path.join(best_src, "txt")
        os.makedirs(txt_dir, exist_ok=True)
        !colmap model_converter --input_path "{best_src}" --output_path "{txt_dir}" --output_type TXT 2>/dev/null

    # Report
    final = MERGED_DIR if os.path.exists(os.path.join(MERGED_DIR, "images.bin")) else os.path.join(COLMAP_DIR, "0")
    if os.path.exists(os.path.join(final, "images.bin")):
        with open(os.path.join(final, "images.bin"), "rb") as f:
            n = struct.unpack("Q", f.read(8))[0]
        with open(os.path.join(final, "points3D.bin"), "rb") as f:
            pts = struct.unpack("Q", f.read(8))[0]
        print(f"\nOptimized COLMAP result: {n} images, {pts} 3D points")
        set_param("merged_images", n)
        set_param("merged_points3d", pts)
    else:
        print("\nNo model available. Use Step 5 to download pre-computed data.")

    
    mark_step("colmap_merged")

# ── Copy COLMAP data to Step 6's expected location ──
INPUT_SPARSE = "/content/gaussian-splatting/input/sparse/0"
os.makedirs(INPUT_SPARSE, exist_ok=True)
final_model = MERGED_DIR if os.path.exists(os.path.join(MERGED_DIR, "images.bin")) else os.path.join(COLMAP_DIR, "0")
for fname in ["cameras.txt", "images.txt", "points3D.txt"]:
    txt_src = os.path.join(os.path.join(final_model, "txt"), fname)
    if os.path.exists(txt_src):
        !cp "{txt_src}" "{INPUT_SPARSE}/{fname}"
        print(f"  Copied {fname} to {INPUT_SPARSE}/")
    elif os.path.exists(os.path.join(final_model, fname)):
        !cp "{os.path.join(final_model, fname)}" "{INPUT_SPARSE}/{fname}"
        print(f"  Copied {fname} (from model root) to {INPUT_SPARSE}/")
    else:
        print(f"  WARNING: {fname} not found in COLMAP output")

---
## Step 5: Download Pre-computed COLMAP Data (~1 min)

Skip COLMAP (Step 4) and download pre-computed camera poses.
Choose the optimized 34-image merged model for best results.
---


In [ ]:
#@title === 5: Download Pre-computed COLMAP Data (~1 min, idempotent) ===
import os, requests

if is_step_done("colmap_downloaded"):
    print("Pre-computed COLMAP data already downloaded (session state). Skipping.")
else:
    INPUT_DIR = "/content/gaussian-splatting/input"
    SPARSE_DIR = os.path.join(INPUT_DIR, "sparse", "0")
    os.makedirs(SPARSE_DIR, exist_ok=True)
    GITHUB_REPO = "kaarthik-balakrishnan/arena-3dgs"
    BASE_URL = f"https://raw.githubusercontent.com/{GITHUB_REPO}/main"

    # Check if already downloaded
    if os.path.exists(os.path.join(SPARSE_DIR, "images.txt")):
        with open(os.path.join(SPARSE_DIR, "images.txt")) as f:
            n = sum(1 for l in f if l.strip() and not l.startswith('#')) // 2
        print(f"COLMAP data already present ({n} images). Skipping.")
    else:
        print("\nWhich COLMAP model to download?")
        print("  [1] 34-image merged model (RECOMMENDED — more registered views)")
        print("  [2] 30-image original model")
        choice = input("Enter 1 or 2 (default: 1): ").strip() or "1"

        if choice == "2":
            files_to_download = [
                "colmap_data/cameras.txt",
                "colmap_data/images.txt",
                "colmap_data/points3D.txt",
            ]
            expected_imgs = 30
        else:
            files_to_download = [
                "colmap_data_optimized/cameras.txt",
                "colmap_data_optimized/images.txt",
                "colmap_data_optimized/points3D.txt",
            ]
            expected_imgs = 34

        local_names = ["cameras.txt", "images.txt", "points3D.txt"]
        for remote, local in zip(files_to_download, local_names):
            url = f"{BASE_URL}/{remote}"
            print(f"Downloading {local}...")
            r = requests.get(url)
            if r.status_code == 200:
                with open(os.path.join(SPARSE_DIR, local), 'w') as f:
                    f.write(r.text)
                print(f"  OK ({len(r.text)/1024:.0f} KB)")
            else:
                print(f"  FAILED (status {r.status_code})")

        set_param("colmap_download_choice", choice)
        set_param("expected_images", expected_imgs)

    # Verify
    with open(os.path.join(SPARSE_DIR, "images.txt")) as f:
        img_lines = [l for l in f if l.strip() and not l.startswith('#')]
        num_images = len(img_lines) // 2
    print(f"\nCOLMAP data: {num_images} registered images (SIMPLE_RADIAL)")
    mark_step("colmap_downloaded")


---
## Step 6: Convert to 3DGS Format (~1 min)

Converts SIMPLE_RADIAL camera model to PINHOLE (required by 3DGS text reader).
Rebuilds binary files from text. Runs every time (fast).
---


In [ ]:
#@title === 6: Convert Data to 3DGS Format (~1 min, idempotent) ===
import os, glob, subprocess, shutil

if is_step_done("data_converted"):
    print("Data already converted (session state). Skipping.")
else:
    # ── Check colmap is installed ──
    if not os.popen("which colmap 2>/dev/null").read().strip():
        raise RuntimeError(
            "colmap not found. Run Cell 2 first to install dependencies."
        )

    INPUT_DIR = "/content/gaussian-splatting/input"
    SPARSE_DIR = os.path.join(INPUT_DIR, "sparse", "0")
    images_dir = os.path.join(INPUT_DIR, "images")

    # Organize images (idempotent)
    if os.path.exists(images_dir) and len(os.listdir(images_dir)) >= 30:
        print(f"Images already organized ({len(os.listdir(images_dir))} files).")
    else:
        os.makedirs(images_dir, exist_ok=True)
        for ext in ['*.jpg', '*.jpeg', '*.png']:
            for f in glob.glob(os.path.join(INPUT_DIR, ext)):
                os.rename(f, os.path.join(images_dir, os.path.basename(f)))
        print(f"Moved {len(os.listdir(images_dir))} images to input/images/")

    # Convert COLMAP to PINHOLE + binary
    required = ["cameras.txt", "images.txt", "points3D.txt"]
    missing = [f for f in required if not os.path.exists(os.path.join(SPARSE_DIR, f))]
    if missing:
        print(f"ERROR: Missing COLMAP data: {missing}. Run Step 5 first.")
    else:
        # Convert SIMPLE_RADIAL -> PINHOLE in cameras.txt
        cam_path = os.path.join(SPARSE_DIR, "cameras.txt")
        with open(cam_path) as f:
            lines = f.readlines()
        modified = False
        with open(cam_path, 'w') as f:
            for line in lines:
                if line.startswith('#') or not line.strip():
                    f.write(line)
                else:
                    parts = line.strip().split()
                    if parts[1] == "SIMPLE_RADIAL":
                        f.write(f"{parts[0]} PINHOLE {parts[2]} {parts[3]} {parts[4]} {parts[4]} {parts[5]} {parts[6]}\n")
                        modified = True
                    else:
                        f.write(line)
        if modified:
            print("  Converted camera model: SIMPLE_RADIAL -> PINHOLE")

        # Rebuild binary files from text
        for fn in ['cameras.bin', 'images.bin', 'points3D.bin']:
            p = os.path.join(SPARSE_DIR, fn)
            if os.path.exists(p):
                os.remove(p)
        bin_dir = "/content/gaussian-splatting/sparse_bin"
        os.makedirs(bin_dir, exist_ok=True)
        result = subprocess.run(
            ["colmap", "model_converter",
             "--input_path", SPARSE_DIR,
             "--output_path", bin_dir,
             "--output_type", "BIN"],
            capture_output=True, text=True
        )
        if os.path.exists(os.path.join(bin_dir, "images.bin")):
            for fn in ["cameras.bin", "images.bin", "points3D.bin"]:
                shutil.copy2(os.path.join(bin_dir, fn), os.path.join(SPARSE_DIR, fn))
            shutil.rmtree(bin_dir, ignore_errors=True)
            print("  Built binary files (cameras.bin, images.bin, points3D.bin)")
        else:
            print(f"WARNING: Binary conversion failed: {result.stderr}")

        # Verify
        with open(os.path.join(SPARSE_DIR, "images.txt")) as f:
            num_images = sum(1 for l in f if l.strip() and not l.startswith('#')) // 2
        print(f"\nReady for training: {num_images} images, PINHOLE model")
        set_param("training_images", num_images)

    mark_step("data_converted")


---
## Step 7: Train 3D Gaussian Splatting

Training uses the enhanced gsplat-based script for faster, more memory-efficient
training without CUDA extension compilation.

**Run order:**
1. Cell 7A: Quick test (3K iters, ~7 min) — verify everything works
2. Cell 7B: Full training (30K iters, ~30 min) — main training run
---


In [ ]:
#@title === 7A: Quick Test (3000 iters, ~7 min) ===
import os, sys

if is_step_done("training_3k"):
    print("Quick test already done (session state). Skipping.")
else:
    from scripts.train_3dgs_enhanced import train as train_3dgs
    from argparse import Namespace
    INPUT_DIR = "/content/gaussian-splatting/input"
    OUTPUT_DIR = "/content/gaussian-splatting/output/quick_test"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    args = Namespace(
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        iterations=3000,
        max_gaussians=500000,
        log_interval=500,
        max_res=1600,
    )
    try:
        train_3dgs(args)
        print("\nQuick test complete! Proceed to full training.")
        mark_step("training_3k")
    except Exception as e:
        print(f"\nERROR: Training failed: {e}")
        import traceback
        traceback.print_exc()



In [ ]:
#@title === 7B: Full Training 30K (~30 min, single run) ===
import os, sys

if is_step_done("training_30k"):
    print("Full training already done (session state). Skipping.")
else:
    # Purge cached modules to ensure fresh import after potential script update
    if 'scripts.train_3dgs_enhanced' in sys.modules:
        del sys.modules['scripts.train_3dgs_enhanced']
    from scripts.train_3dgs_enhanced import train as train_3dgs
    from argparse import Namespace
    INPUT_DIR = "/content/gaussian-splatting/input"
    OUTPUT_DIR = "/content/gaussian-splatting/output/arena_3dgs"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    args = Namespace(
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        iterations=30000,
        max_gaussians=500000,
        log_interval=1000,
        max_res=1600,
    )
    try:
        train_3dgs(args)
        print("\nFull training complete!")
        mark_step("training_30k")
    except Exception as e:
        print(f"\nERROR: Training failed: {e}")
        import traceback
        traceback.print_exc()



In [ ]:
#@title === 7C (Obsolete): Use 7B above for single-run full training ===
print("Cell 7C is obsolete. Use Cell 7B for full 30K training in a single run.")
print("The enhanced training script uses gsplat and runs ~30 min on T4.")

In [ ]:
#@title === 7D (Obsolete): Use 7B above for single-run full training ===
print("Cell 7D is obsolete. Use Cell 7B for full 30K training in a single run.")
print("The enhanced training script uses gsplat and runs ~30 min on T4.")


In [ ]:
#@title === 7E (Obsolete): Use 7B above for single-run full training ===
print("Cell 7E is obsolete. Use Cell 7B for full 30K training in a single run.")
print("The enhanced training script uses gsplat and runs ~30 min on T4.")

---
## Step 8: Export & Validate for Unity (~1 min)
---


In [ ]:
#@title === 8A: Export Point Cloud (~1 min) ===
import os
from google.colab import files

if is_step_done("exported_ply"):
    print("PLY already exported (session state). Skipping.")
else:
    PLY_DST = "/content/arena_3dgs_pointcloud.ply"

    candidates = [
        "/content/gaussian-splatting/output/arena_3dgs/arena_3dgs.ply",
        "/content/gaussian-splatting/output/quick_test/arena_3dgs.ply",
    ]

    found_ply = None
    for p in candidates:
        if os.path.exists(p):
            found_ply = p
            print(f"Found: {p}")
            break

    if found_ply:
        !cp "{found_ply}" "{PLY_DST}"
        size_mb = os.path.getsize(PLY_DST) / (1024 * 1024)
        print(f"\nPoint cloud: {PLY_DST} ({size_mb:.1f} MB)")
        save_to_drive(PLY_DST, "arena_3dgs_pointcloud.ply")
        print(f"Backed up to Drive: {checkpoint_path('arena_3dgs_pointcloud.ply')}")
        print("\nDownloading to your computer...")
        files.download(PLY_DST)
    else:
        print("No trained model found. Run training cells (7A or 7B) first.")

    mark_step("exported_ply")


In [ ]:
#@title === 8B: Validate PLY for Unity (~1 min) ===
import os

if is_step_done("validated_ply"):
    print("PLY already validated (session state). Skipping.")
else:
    PLY_PATH = "/content/arena_3dgs_pointcloud.ply"

    if not os.path.exists(PLY_PATH):
        print("No PLY found. Run 8A first.")
    else:
        from plyfile import PlyData
        import numpy as np

        ply = PlyData.read(PLY_PATH)
        data = ply['vertex'].data
        n = len(data)
        print(f"Gaussians: {n:,}")

        required = ['x', 'y', 'z', 'f_dc_0', 'f_dc_1', 'f_dc_2',
                    'opacity', 'scale_0', 'scale_1', 'scale_2',
                    'rot_0', 'rot_1', 'rot_2', 'rot_3']
        missing = [p for p in required if p not in data.dtype.names]
        if missing:
            print(f"\nWARNING: Unity requires: {missing}")
        else:
            print("\nUnity format: OK")

        xyz = np.stack([data['x'], data['y'], data['z']], axis=1)
        print(f"Bounds: X[{xyz[:,0].min():.1f}, {xyz[:,0].max():.1f}] "
              f"Y[{xyz[:,1].min():.1f}, {xyz[:,1].max():.1f}]")

        opacities = data['opacity']
        visible = (np.array(opacities) > 0.01).sum()
        print(f"Visible Gaussians (>0.01 opacity): {visible:,}")
        print(f"File size: {os.path.getsize(PLY_PATH) / 1024**2:.1f} MB")
        set_param("num_gaussians", int(n))
        set_param("ply_size_mb", round(os.path.getsize(PLY_PATH) / 1024**2, 1))

    print("\n" + "=" * 50)
    print("  VIEWING OPTIONS")
    print("=" * 50)
    print("\n1. SuperSplat (no install, web):")
    print("     https://supersplat.com/")
    print("\n2. Unity walkthrough (best):")
    print("     Clone: https://github.com/aras-p/UnityGaussianSplatting")
    print("     Drop PLY into Assets/GaussianAssets/")
    print("     WASD + mouse-look controls")
    print("\n3. Local decompression viewer (lightweight, no GPU):")
    print("     python3 scripts/decompress_splat.py compressed.splat")
    print("     Drag to orbit, scroll to zoom, R=reset, Q=quit")

    mark_step("validated_ply")


In [ ]:
#@title === 8C: Compress Model for Local Viewer (~1 min) ===
import urllib.request
import subprocess, sys, os

if is_step_done("compressed"):
    print("Compression already done (session state). Skipping.")
else:
    PLY_PATH = "/content/arena_3dgs_pointcloud.ply"
    if not os.path.exists(PLY_PATH):
        print("No PLY found. Run 8A first.")
    else:
        SCRIPT = "/content/compress_splat.py"
        if not os.path.exists(SCRIPT):
            url = "https://raw.githubusercontent.com/kaarthik-balakrishnan/arena-3dgs/main/scripts/compress_splat.py"
            urllib.request.urlretrieve(url, SCRIPT)
            print("Downloaded compress_splat.py")

        # Determine quality from params or default to medium
        quality = get_param("compress_quality", "medium")
        print(f"Running compression (quality={quality})...")
        result = subprocess.run(
            [sys.executable, SCRIPT, PLY_PATH, "--quality", quality, "--output-dir", "/content"],
            capture_output=True, text=True
        )
        print(result.stdout)
        if result.returncode != 0:
            print("STDERR:", result.stderr)
        else:
            import glob
            splats = glob.glob("/content/*.splat")
            if splats:
                splat = splats[-1]
                size_mb = os.path.getsize(splat) / (1024 * 1024)
                print(f"\nCompressed: {splat} ({size_mb:.1f} MB)")
                set_param("compressed_size_mb", round(size_mb, 1))
                print("\nDownload the .splat file to your computer, then view:")
                print("  python3 scripts/decompress_splat.py path/to/arena_3dgs_compressed.splat")
                from google.colab import files
                files.download(splat)

    mark_step("compressed")


---
## Appendix: Troubleshooting

| Problem | Solution |
|---------|----------|
| **train_3dgs_enhanced.py not found** | Re-run Cell 2 to download from GitHub |
| **Session disconnected** | Re-open notebook → run Cell 1 → choose "Continue" → run remaining cells |
| **CUDA out of memory** | Reduce `max_gaussians` or `max_res` in Cell 7B. |
| **COLMAP produces 0 images** | Download pre-computed data (Step 5). |
| **Drive restore failed** | Check `MyDrive/arena_3dgs/` exists and has files. |
| **Training segment needs re-run** | It will resume from the last checkpoint automatically. |
| **"Start fresh" still skips steps** | Delete `MyDrive/arena_3dgs/session_state.json` from Drive and re-run Cell 1. |

### Session State

Progress is tracked in `MyDrive/arena_3dgs/session_state.json`. This file records:
- Which steps have been completed
- Parameters (image counts, model sizes, etc.)
- Session creation and update timestamps

To force a complete restart from scratch:
1. Run Cell 1 and enter `n` when asked to continue, OR
2. Delete `session_state.json` from Drive manually

### Reference (PIPELINE.md)

Full implementation details, academic references, and efficiency notes:
[PIPELINE.md on GitHub](https://github.com/kaarthik-balakrishnan/arena-3dgs/blob/main/PIPELINE.md)

### Checkpoint locations (in Google Drive):
- `MyDrive/arena_3dgs/session_state.json` — Session progress tracker
- `MyDrive/arena_3dgs/database.db` — COLMAP features + matches
- `MyDrive/arena_3dgs/sparse_model` — COLMAP reconstruction
- `output/arena_3dgs/arena_3dgs.ply` — Trained model PLY file
- `MyDrive/arena_3dgs/arena_3dgs_pointcloud.ply` — Final PLY export

---
